# Consultas SQL · cuaderno del proyecto
**Equipo E9 · Empresas que exportan desde Antioquia · Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga la muestra de la DIAN.
3. La celda 3 cuenta las filas: la primera **cifra de control**.
4. Sesión 3: las dos consultas sobre la muestra.
5. Al final: consultas sobre la BD depurada completa, también desde GitHub.

In [ ]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

## 2. Carga la muestra

La muestra son las primeras 1.000 líneas de la base de exportaciones de la DIAN de septiembre de 2026, con 70 de sus 72 columnas (se quitaron el nombre y el domicilio del comprador en el exterior porque pueden ser de una persona). Está en `datos/originales/`.

**Opción A (recomendada).** Deja el enlace que viene. **Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo.

Deben salir **1.000 filas y 70 columnas**.

In [ ]:
# Celda 2 · Cargar la muestra
ENLACE = "https://raw.githubusercontent.com/sebas9822-beep/Grupo_E9_Empresas_Exportando/main/datos/originales/muestra_dian_exportaciones_2026_09.csv"

if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

## 3. ¿Cuántas filas hay?

Compara con lo que dice el diccionario en «Filas en la muestra» (1.000). Si no coincide, anótalo en la bitácora.

In [ ]:
# Celda 3 · Contar filas
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

## 4. Los nombres exactos de las columnas

Cópialos de aquí. Los valores de texto van entre **comillas simples**: `'ANTIOQUIA'`.

In [ ]:
# Celda 4 · Columnas de la muestra
list(datos.columns)

---
# Sesión 3 · Dos consultas sobre la muestra

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después las que deja el filtro (mercancía de origen Antioquia); al final mira algunas.

Cifra de control: 1.000 filas en total y 140 de origen Antioquia.

In [ ]:
# Celda 5 · Consulta 1a: filas antes del filtro
sql("""
SELECT COUNT(*) AS filas_total
FROM datos
""")

In [ ]:
# Celda 6 · Consulta 1b: filas después del filtro
sql("""
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE REGION_DE_ORIGEN = 'ANTIOQUIA'
""")

In [ ]:
# Celda 7 · Consulta 1c: mirar las filas filtradas, solo las columnas que necesitas
sql("""
SELECT NIT_EXPORTADOR, RAZON_SOCIAL_EXPORTADOR, SUBPARTIDA, PAIS_DESTINO_FINAL, VALOR_FOB_USD
FROM datos
WHERE REGION_DE_ORIGEN = 'ANTIOQUIA'
LIMIT 10
""")

**Consulta 2 · Resumir.** Cuenta cuántas líneas hay por departamento de origen de la mercancía.

**Cifra de control:** la suma de los grupos debe dar el total de la celda 5 (1.000). En la muestra, Bogotá 468, Cundinamarca 350 y Antioquia 140 son los tres primeros.

In [ ]:
# Celda 8 · Consulta 2: resumen por grupo
sql("""
SELECT REGION_DE_ORIGEN, COUNT(*) AS filas
FROM datos
GROUP BY REGION_DE_ORIGEN
ORDER BY filas DESC
""")

In [ ]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql("""
SELECT SUM(filas) AS suma_de_los_grupos
FROM (
    SELECT REGION_DE_ORIGEN, COUNT(*) AS filas
    FROM datos
    GROUP BY REGION_DE_ORIGEN
)
""")

---
# BD depurada · consultas sobre los datos limpios

Los datos limpios están en `datos/limpios/`. Aquí se cargan dos piezas desde GitHub, sin descargar nada a mano:

- el **directorio de exportadores antioqueños** (CSV, una fila por empresa), y
- las **exportaciones de septiembre de 2026** de la tabla de hechos (parquet) con la clasificación de productos.

Para trabajar con todos los meses, arma la BD completa en tu computador con `python crear_bd_depurada.py` (ver el README).

In [ ]:
# Celda 10 · Directorio de exportadores antioqueños
directorio = cargar("https://raw.githubusercontent.com/sebas9822-beep/Grupo_E9_Empresas_Exportando/main/datos/limpios/directorio_exportadores_antioquenos.csv")
print(f"{len(directorio)} empresas")  # cifra de control: 3195 (2875 con vínculo A y 320 con vínculo B)
sql("""
SELECT VINCULO_ANTIOQUIA, COUNT(*) AS empresas
FROM directorio
GROUP BY VINCULO_ANTIOQUIA
ORDER BY VINCULO_ANTIOQUIA
""")

In [ ]:
# Celda 11 · Las 10 empresas antioqueñas que más exportaron en 2026 (enero a septiembre)
sql("""
SELECT RANKING_ANTIOQUIA_FOB, RAZON_SOCIAL, SECTOR_EXPORTADOR_PRINCIPAL,
       ROUND(FOB_USD_2026 / 1e6, 1) AS fob_2026_musd, TOP3_PAISES_DESTINO
FROM directorio
ORDER BY FOB_USD_2026 DESC NULLS LAST
LIMIT 10
""")
# La primera es COMERCIALIZADORA INTERNACIONAL ESLOP SAS, con USD 1.434,4 M

In [ ]:
# Celda 12 · Exportaciones de septiembre de 2026 por sector (hechos + subpartidas)
BASE = "https://raw.githubusercontent.com/sebas9822-beep/Grupo_E9_Empresas_Exportando/main/datos/limpios/bd_depurada"
hechos = pd.read_parquet(f"{BASE}/hechos_exportaciones/hechos_2026_09.parquet")
subpartidas = pd.read_parquet(f"{BASE}/dim_subpartida.parquet")
print(f"{len(hechos)} líneas en septiembre de 2026")  # cifra de control: 115049
sql("""
SELECT s.SECTOR_EXPORTACION, ROUND(SUM(h.VALOR_FOB_USD) / 1e6, 1) AS fob_musd
FROM hechos AS h
JOIN subpartidas AS s ON s.SUBPARTIDA = h.SUBPARTIDA
GROUP BY s.SECTOR_EXPORTACION
ORDER BY fob_musd DESC
""")
# Cifra de control: 3.060,2 + 1.273,4 + 1.077,1 = USD 5.410,7 M, el total del mes

## Guarda tus consultas en GitHub

En el repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`07_mi_consulta.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas líneas de la muestra son de origen Antioquia?
-- Datos: datos/originales/muestra_dian_exportaciones_2026_09.csv (1.000 filas)
-- Cifra de control: 1.000 en total, 140 de origen Antioquia
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE REGION_DE_ORIGEN = 'ANTIOQUIA';
```

Las consultas que ya están en `consultas/` sirven de ejemplo.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.